# ScriptureBigData — Getting Started

Treat the whole of Scripture (Hebrew Old Testament, Greek New Testament, and the Greek Septuagint) as **big data**: search it, analyse it, visualise it, and connect words and passages across it — in the **original languages**.

This notebook uses the small committed **offline sample** so it runs with no downloads. For the full corpus run `sbd ingest --full` and open that database instead.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
from scripturebigdata import Scripture

# Offline demo corpus (Genesis 1, Psalm 23, Isaiah 7 & 53, Jonah 1, John 1, Matthew 1 & 5, 1 John 4)
bible = Scripture.from_sample()
bible.stats()

## 1. Search words in the original languages

Search by **lemma** (dictionary form), exact **surface** form, **Strong's** number, English **gloss**, **part of speech**, or **morphology**. Matching ignores Hebrew vowel points and Greek accents.

In [ ]:
# Every occurrence of ἀγάπη ('love') — any inflected form
for h in bible.search('ἀγάπη', field='lemma'):
    print(f"{h.reference:14} {h.surface:10} {h.strongs} {h.gloss}")

In [ ]:
# Word-by-word interlinear of John 1:1
for t in bible.interlinear('John', 1, 1):
    print(f"{t['position']:>2}. {t['surface']:12} {t['lemma']:10} {t['strongs']:6} {t['gloss'][:24]:24} {t['morph_desc']}")

## 2. Connect the languages: Hebrew ↔ Septuagint

See a verse in every edition side by side — the Hebrew of Genesis 1:1 next to how the Greek Septuagint translated it.

In [ ]:
for edition, tokens in bible.alignment('Gen', 1, 1).items():
    print(edition.ljust(12), ' '.join(t['surface'] for t in tokens))

## 3. Connect passages: find quotations across Scripture

The cross-reference engine finds other verses that share a run of words. This is exactly how a **New Testament quotation of the Septuagint** surfaces automatically — Matthew 1:23 quotes the Greek of Isaiah 7:14.

In [ ]:
for x in bible.cross_references('SBLGNT', 'Matt', 1, 23, min_n=4):
    print(f"{x['reference']:16} {x['edition']:12} ({x['words']} words)  «{x['phrase']}»")

## 4. Find patterns: frequency, distribution, co-occurrence

In [ ]:
# Most frequent lemmas in the New Testament sample
for r in bible.frequency(by='lemma', corpus='NT', top=8):
    print(f"{r['n']:>4}  {r['lemma']:10} {r['strongs']:6} {r['gloss']}")

print('\nDistribution of the divine name (H3068) across books:')
for d in bible.distribution('H3068'):
    print(f"  {d['book']:6} {'█' * d['n']} {d['n']}")

## 5. Visualise

`scripturebigdata.viz` renders dependency-free SVG (and optional matplotlib PNGs).

In [ ]:
from IPython.display import SVG, display
from scripturebigdata import viz
freq = bible.frequency(by='lemma', corpus='NT', top=10)
display(SVG(viz.bar_svg(freq, title='Most frequent NT lemmas (sample)')))

## 6. Updatable text: register a manuscript variant

The base text is versioned, and newly-discovered readings can be layered on as a **textual apparatus** without altering the base — satisfying “the text can be updated as variants are discovered.”

In [ ]:
bible.add_variant(edition='SBLGNT', book='John', chapter=1, verse=18,
                  witness='Codex Sinaiticus', reading='μονογενὴς θεός',
                  note="'only-begotten God' vs 'only-begotten Son'")
bible.variants('John', 1, 18)